In [1]:
import yfinance as yf
import sys
import pandas as pd
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, InputLayer, Dropout
from tensorflow.keras.optimizers import Adam
from collections import deque
import time
import os
import pandas_ta as ta
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timezone
# from pymongo import MongoClient
# from bson import ObjectId
import numpy as np
from fredapi import Fred
from dateutil.relativedelta import relativedelta
from curl_cffi import requests
import plotly.graph_objects as go

C:\Users\micha\anaconda32\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.0 is exactly one major version older than the runtime version 6.30.2 at yfinance/pricing.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(


In [2]:
# client = MongoClient('mongodb://localhost:27017/')  
# print('Connected to MongoDB')
# db = client['nlp-based-sentiment-analysis']
# prices = db['prices']

In [3]:
STARTING_DATE = "2024-05-01"
ENDING_DATE = "2025-05-01"
GAMMA = 0.95
EPSILON = 1.0
MODEL_DESIGN = "64/64"
EPISODES = 2
MEMORY_LENGTH = 100
BATCH_SIZE = 64
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
TARGET_UPDATE = 20
LEARNING_RATE = 0.0005
LABEL_BULLISH = "Bullish"
LABEL_BEARISH = "Bearish"
LABEL_NEUTRAL = "Neutral"
TEST = False

In [4]:
def categorize_rsi(rsi_series):
    conditions = [
        rsi_series < 30,
        rsi_series > 70,
    ]
    choices = [LABEL_BULLISH, LABEL_BEARISH]
    return np.select(conditions, choices, default=LABEL_NEUTRAL)

def generate_model(state_size, action_size):
    layers = MODEL_DESIGN.split("/")
    model = Sequential()
    model.add(InputLayer(input_shape=(state_size,), name="InputLayer"))
    for index, units in enumerate(layers):
        model.add(Dense(units=units, activation="relu", name=f"HiddenLayer{index}"))
        model.add(Dropout(0.2))
    model.add(Dense(units=action_size, activation='linear', name="OutputLayer"))
    return model

def calculate_barriers(price, volatility, Fu, Fl):
    upper = price + price * volatility * Fu
    lower = price - price * volatility * Fl
    return upper, lower

def get_best_params_for_day(results, day):
    day = pd.to_datetime(day)
    for res in results:
        if res['start'] <= day < res['end']:
            return res['best_params']
    return None  # If day is outside all defined windows

def get_label_for_a_day(labels, day):
    day = pd.to_datetime(day)
    for label in labels:
        if label['start'] <= day < label['end']:
            return label['label']
    return None  # Ensure it's always a tuple with 2 values

def get_prev_label_for_a_day(labels, day):
    day = pd.to_datetime(day)
    for label in labels:
        if label['start'] <= day < label['end']:
            return label['prev_label']
    return None  # Ensure it's always a tuple with 2 values

def assign_labels(data, fu, fl, vt, best_params_for_day=None):
    barriers = None
    start_day = None
    end_day = None
    prev_label = None
    labels = []
    for index, day in data[:-1].iterrows():
        if best_params_for_day != None:
            params = get_best_params_for_day(best_params_for_day, day["date"])
            if params is not None:
                fu, fl, vt = params
            else:
                fu, fl, vt = 1.0, 1.0, 10
        if barriers == None:
            barriers = calculate_barriers(day["Open"], day["Volatility"], fu, fl)
            high_barrier, low_barrier = barriers
            start_day = day["date"]
            days = 0
        days += 1
        if day["High"] > barriers[0]:
            labels.append({"start": start_day, "end": data["date"][index+1], "label": LABEL_BULLISH, "prev_label": prev_label})
            start_day = None
            end_day = None
            barriers = None
            prev_label = LABEL_BULLISH
        elif day["Low"] < barriers[1]:
            labels.append({"start": start_day, "end": data["date"][index+1], "label": LABEL_BEARISH, "prev_label": prev_label})
            start_day = None
            end_day = None
            barriers = None
            prev_label = LABEL_BEARISH
        elif start_day != None and end_day == None and days == vt:
            labels.append({"start": start_day, "end": data["date"][index+1], "label": LABEL_NEUTRAL, "prev_label": prev_label})
            start_day = None
            end_day = None
            barriers = None
            prev_label = LABEL_NEUTRAL
            days = 0
    return labels

def simulate_returns(df, labels, Vt):
    returns = []
    for i in range(len(df) - Vt):
        start_price = df['Close'].iloc[i]
        end_price = df['Close'].iloc[i+Vt]
        label = get_label_for_a_day(labels, df['date'].iloc[i])
        if label == LABEL_BULLISH:
            ret = (end_price - start_price) / start_price
        elif label == LABEL_BEARISH:
            ret = (start_price - end_price) / start_price
        else:
            ret = 0
        returns.append(ret)
    returns.extend([0] * Vt)
    return np.array(returns)

def optimize_sharpe(df, Fu_range, Fl_range, Vt_range, rf=0.04):
    df = df.sort_values("date").reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])
    
    start_date = df["date"].min()
    end_date = df["date"].max()
    window_results = []

    current_start = start_date

    while current_start + relativedelta(months=6) <= end_date:
        current_end = current_start + relativedelta(months=6)
        window_df = df[(df["date"] >= current_start) & (df["date"] < current_end)].reset_index(drop=True)

        best_sr = -np.inf
        best_params = None

        for Fu in Fu_range:
            for Fl in Fl_range:
                for Vt in Vt_range:
                    if len(window_df) < Vt + 1:
                        continue
                    labels = assign_labels(window_df, Fu, Fl, Vt)
                    rets = simulate_returns(window_df, labels, Vt)
                    mean_ret = np.mean(rets) * 252
                    std_ret = np.std(rets) * np.sqrt(252)
                    if std_ret > 0:
                        sr = (mean_ret - rf) / std_ret
                        if sr > best_sr:
                            best_sr = sr
                            best_params = (Fu, Fl, Vt)
        
        window_results.append({
            "start": current_start,
            "end": current_end,
            "best_params": best_params,
            "sharpe": best_sr
        })

        current_start += relativedelta(months=6)

    return window_results

# Run optimization
Fu_range = np.arange(0.5, 3.1, 0.5)
Fl_range = np.arange(0.5, 3.1, 0.5)
Vt_range = range(8, 16)

In [5]:
def draw_lines(fig, start_day, data, barriers, index, label):
    fig.add_shape(type="line", x0=start_day, x1=data["date"][index+1], y0=barriers[0], y1=barriers[0], line=dict(color="green"), name="Upper Barrier")
    fig.add_shape(type="line", x0=start_day, x1=data["date"][index+1], y0=barriers[1], y1=barriers[1], line=dict(color="red"), name="Lower Barrier")
    fig.add_shape(type="line", yref="paper", x0=data["date"][index+1], x1=data["date"][index+1], y0=0, y1=1, line=dict(color="gray", dash="dot"), name="Vertical Barrier")
    fig.add_shape(
        type="rect",
        x0=start_day,
        x1=data["date"][index+1],
        y0=0,
        y1=1,
        yref="paper",
        fillcolor="rgba(0, 255, 0, 0.1)" if label == LABEL_BULLISH else "rgba(255, 0, 0, 0.1)" if label == LABEL_BEARISH else "rgba(128,128,128,0.1)",  # light green with transparency
        line=dict(width=0),
        layer="below"  # draw behind the data
    )

def draw_barriers(data, fig, fu, fl, vt, best_params_for_day=None):
    barriers = None
    start_day = None
    end_day = None
    label = None
    next_label = None
    labels = []
    for index, day in data[:-1].iterrows():
        if best_params_for_day != None:
            params = get_best_params_for_day(best_params_for_day, day["date"])
            if params is not None:
                fu, fl, vt = params
                # Proceed with your logic using fu, fl, vt
            else:
                # Handle missing params (e.g., use default values or skip the day)
                fu, fl, vt = 1.0, 1.0, 10  # Example default
        if label != next_label:
            label = next_label
        if barriers == None:
            barriers = calculate_barriers(day["Open"], day["Volatility"], fu, fl)
            high_barrier, low_barrier = barriers
            start_day = day["date"]
            days = 0
        days += 1
        if day["High"] > barriers[0]:
            draw_lines(fig, start_day, data, barriers, index, LABEL_BULLISH)
            start_day = None
            end_day = None
            barriers = None
            next_label = LABEL_BULLISH
        elif day["Low"] < barriers[1]:
            draw_lines(fig, start_day, data, barriers, index, LABEL_BEARISH)
            start_day = None
            end_day = None
            barriers = None
            next_label = LABEL_BEARISH
        elif start_day != None and end_day == None and days == vt:
            draw_lines(fig, start_day, data, barriers, index, LABEL_NEUTRAL)
            start_day = None
            end_day = None
            barriers = None
            next_label = LABEL_NEUTRAL
            days = 0

In [6]:
class Agent():
    def __init__(self, action_size, state_size, gamma, epsilon, epsilon_min, epsilon_decay, model_id = 0):
        self.action_size = action_size
        self.state_size = state_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=MEMORY_LENGTH)
        self.model = self.create_model()
        self.target_model = clone_model(self.model)
        self.optimizer = Adam(learning_rate=LEARNING_RATE) 
        self.step_counter = 0

    def create_model(self):
        return generate_model(self.state_size, self.action_size)

    def act(self, state):
        if random.uniform(0,1) < self.epsilon:
            rand_action = random.randrange(self.action_size)
            return rand_action
        q_values = self.model.predict([state], verbose=0)
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, new_state, done):
        self.memory.append((state, action, reward, new_state, done))
        
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())
    
    def replay(self, step):
        if len(self.memory) < BATCH_SIZE:
            return
        minibatch = random.sample(self.memory, BATCH_SIZE)
        states = np.array([sample[0] for sample in minibatch], dtype=np.float32).reshape(BATCH_SIZE, self.state_size)
        actions = np.array([sample[1] for sample in minibatch], dtype=np.int32)
        rewards = np.array([sample[2] for sample in minibatch], dtype=np.float32)
        new_states = np.array([sample[3] for sample in minibatch], dtype=np.float32).reshape(BATCH_SIZE, self.state_size)
        done = np.array([sample[4] for sample in minibatch])
        best_action_indices = np.argmax(self.model.predict([new_states], verbose=0), axis=1)
        target = rewards + (1 - done) * self.gamma * self.target_model.predict([new_states], verbose=0)[np.arange(BATCH_SIZE), best_action_indices]
        with tf.GradientTape() as tape:
            current_Q_values = self.model([states], training=True)
            action_mask = tf.one_hot(actions, current_Q_values.shape[1])
            predicted_Q_values = tf.reduce_sum(current_Q_values * action_mask, axis=1)
            loss = tf.keras.losses.MeanSquaredError()(target, predicted_Q_values)
        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.model.trainable_variables))
        self.step_counter += 1
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        if self.step_counter % TARGET_UPDATE == 0:
            self.step_counter = 0
            self.update_target_model()

In [7]:
class Environment():
    def __init__(self, data, data_scaled):
        self.action_size = 2
        self.state_size = np.array(data_scaled).shape[1]
        self.data = data
        self.data_scaled = data_scaled
        self.offset = 0
        self.steps = len(data)
    
    def step(self, action):
        self.offset = self.offset + 1
        index =  self.offset
        new_state = self.data_scaled[index]
        price_change = self.data[index][0] - self.data[index - 1][0]
#         if self.data[index][1] == LABEL_BULLISH and action == 0:
#             reward = price_change
#         elif self.data[index][1] == LABEL_BEARISH and action == 1:
#             reward = -price_change
#         elif self.data[index][1] == LABEL_NEUTRAL and action == 2:
#             reward = 0.1  # small positive feedback
#         else:
#             reward = -abs(price_change)  # penalize wrong action based on real movement
#         if self.data[index][1] == LABEL_BULLISH:
#             reward = 1 if action == 0 else 0
#         elif self.data[index][1] == LABEL_BEARISH:
#             reward = 1 if action == 1 else 0
#         else:
#             reward = 0  # or optionally: small penalty for action, e.g., reward = -1
        reward = price_change if action == 0 else -price_change # Buy=0, Sell=1
        if reward > 0:
            reward = 10
        else:
            reward = -10
#         reward = price_change if action == 0 else -price_change if action == 1 else 0 # Buy=0, Sell=1, Hold=2
        done = self.offset == len(self.data) - 1
        return new_state, reward, done
    
    def reset(self):
        self.offset = 0
        return self.data_scaled[self.offset]

In [8]:
class DQNAlgorithm():
    def __init__(self, data, data_scaled, episodes, gamma, epsilon, epsilon_min, epsilon_decay):
        self.episodes = episodes
        self.data = data
        self.data_scaled = data_scaled
        self.env = Environment(data, data_scaled)
        self.steps = self.env.steps
        self.agent = Agent(self.env.action_size, self.env.state_size, gamma, epsilon, epsilon_min, epsilon_decay)
        self.best_model_win_rate = 0

    def test(self):
        state = self.env.reset()
        rewards = 0
        profits = []
        balances = []
        sum_spend = 0
        sum_earned = 0
        purchases = 0
        sells = 0
        successes = 0
        fails = 0
        actions = []
        true_labels = []
        for step in range(self.steps):
            print(f"Step {step}")
            print(f"Current price {self.data[self.env.offset][0]}")
            print(f"Current label {self.data[self.env.offset][1]}")
            print(f"Offset: {self.env.offset}")
            print(f"State: {state}")
            action = self.agent.act(state)
            print(f"Action: {action}")
            if action == 0:  # Buy
                sum_spend += self.data[self.env.offset][0]
                purchases += 1
            elif action == 1:  # Sell
                sum_earned += self.data[self.env.offset][0]
                sells += 1
            new_state, reward, done = self.env.step(action)
            if reward > 0:
                successes += 1
                print("Success")
            elif reward < 0:
                fails += 1
                print("Fail")
            profits.append(reward)
            balances.append(sum_earned - sum_spend)
            print(f"Next label {self.data[self.env.offset][1]}")
            print(f"Next price {self.data[self.env.offset][0]}")
            rewards += reward
            actions.append(action)
            true_action = 0 if reward > 0 else 1
            true_labels.append(true_action)
            print(f"New state: {new_state}")
            print(f"Reward: {reward}")
            self.agent.remember(state, action, reward, new_state, done)
            print(f"Memory length: {len(self.agent.memory)}")
            print("\n")
            if done == True:
                print("End")
                cumulative_returns = np.cumsum(profits).tolist()
                total_trades = len(profits)
                balance = sum_earned - sum_spend
                sharpe = sharpe_ratio(profits)
                sortino = sortino_ratio(profits)
                mdd = max_drawdown(balances)
                pf = profit_factor(profits)
                total_profit = sum(profits)
                total_trade = purchases + sells
                profit_per_trade = total_profit / total_trade if total_trade > 0 else 0
                win_rate = (successes / total_trade) * 100 if total_trade > 0 else 0
                print(f"Sharpe Ratio: {sharpe}")
                print(f"Sortino Ratio: {sortino}")
                print(f"Maximum Drawdown: {mdd}")
                print(f"Profit Factor: { pf}")
                print(f"Final Balance: {balance}")
                print(f"Profit Per Trade: {profit_per_trade}")
                print(f"Win Rate: {win_rate}")
                print(f"Total Profit: {total_profit}")
                print(f"Successes: {successes}, Fails: {fails}")
                print(f"Buy Actions: {purchases}, Sell Actions: {sells}")
                print(f"Cumulative Returns: {cumulative_returns}")
                break
            self.agent.replay(step)
            state = new_state

    def run(self):
        for episode in range(self.episodes):
            state = self.env.reset()
            profits = []
            balances = []
            sum_spend = 0
            sum_earned = 0
            purchases = 0
            sells = 0
            successes = 0
            fails = 0
            start_time = time.time()
            for step in range(self.steps):
                action = self.agent.act(state)
                new_state, reward, done = self.env.step(action)
                if reward > 0:
                    successes += 1
                elif reward < 0:
                    fails += 1
                if action == 0:  # Buy
                    sum_spend += self.data[self.env.offset - 1][0]
                    purchases += 1
                elif action == 1:  # Sell
                    sum_earned += self.data[self.env.offset - 1][0]
                    sells += 1
                self.agent.remember(state, action, reward, new_state, done)
                balances.append(sum_earned - sum_spend)
                profits.append(reward)
                if done == True:
                    mdd = max_drawdown(balances)
                    total_trade = purchases + sells
                    win_rate = (successes / total_trade) * 100 if total_trade > 0 else 0
                    cumulative_returns = np.cumsum(profits).tolist()
                    print(f"Episode {episode + 1}/{self.episodes}, Win Rate: {win_rate}, Max Drawdown: {mdd}, Cum Returns: {cumulative_returns[-1]}")
                    if win_rate > self.best_model_win_rate:
                        self.best_model_win_rate = win_rate
                        self.best_model = self.agent.model
                    end_time = time.time()
                    print(f"Elapsed time: {end_time - start_time}\n")
                    break
                self.agent.replay(step)
                state = new_state
        return self.best_model

In [9]:
def max_drawdown(balances):
    peak = balances[0]
    max_dd = 0
    for balance in balances:
        peak = max(peak, balance)
        drawdown = (peak - balance) / peak if peak > 0 else 0
        max_dd = max(max_dd, drawdown)
    return max_dd

def sharpe_ratio(returns, risk_free_rate=0.0):
    if len(returns) < 2:
        return 0  # Avoid division by zero
    excess_returns = np.array(returns) - risk_free_rate
    return np.mean(excess_returns) / np.std(excess_returns) if np.std(excess_returns) > 0 else 0

def sortino_ratio(returns, risk_free_rate=0.0):
    excess_returns = np.array(returns) - risk_free_rate
    downside_returns = excess_returns[excess_returns < 0]
    downside_std = np.std(downside_returns) if len(downside_returns) > 0 else 0
    return np.mean(excess_returns) / downside_std if downside_std > 0 else 0

def profit_factor(profits):
    total_profit = sum(p for p in profits if p > 0)
    total_loss = abs(sum(p for p in profits if p < 0))
    return total_profit / total_loss if total_loss > 0 else float('inf')

def evaluate_model(original_data, scaled_data, mod):
    profits = []
    balances = []
    sum_spend = 0
    sum_earned = 0
    purchases = 0
    sells = 0
    successes = 0
    fails = 0
    predictions = []
    true_labels = []
    
    for step, element in enumerate(original_data[:-1]):
        index = step
        q_values = mod.predict([scaled_data[index]], verbose=0)
        action = np.argmax(q_values[0])
        index += 1
        next_price_change = original_data[index][0] - original_data[index - 1][0]
        reward = next_price_change if action == 0 else -next_price_change
        if reward > 0:
            successes += 1
        elif reward < 0:
            fails += 1
        if action == 0:  # Buy
            sum_spend += original_data[index - 1][0]
            purchases += 1
        elif action == 1:  # Sell
            sum_earned += original_data[index - 1][0]
            sells += 1

        true_action = action if reward > 0 else 0 if action == 1 else 1
        predictions.append(action)
        true_labels.append(true_action)

        profits.append(reward)
        balances.append(sum_earned - sum_spend)   
    
    total_trades = len(profits)
    balance = sum_earned - sum_spend
    cumulative_returns = np.cumsum(profits).tolist()
    
    # Metrics Calculation
    sharpe = sharpe_ratio(profits)
    sortino = sortino_ratio(profits)
    mdd = max_drawdown(balances)
    pf = profit_factor(profits)
    total_profits = sum(profits)
    total_trades = purchases + sells
    profit_per_trade = total_profits / total_trades if total_trades > 0 else 0
    win_rate = (successes / total_trades) * 100 if total_trades > 0 else 0

    return {
        "# Purchases": purchases,
        "# Sells": sells,
        "Successes": successes,
        "Fails": fails,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Maximum Drawdown": mdd,
        "Profit Factor": pf,
        "Final Balance": balance,
        "Profit Per Trade": profit_per_trade,
        "Win Rate": win_rate,
        "Total Profits": total_profits,
        "Cumulative Returns": cumulative_returns
    }

In [10]:
def add_months(date_str: str, months: int) -> str:
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")  # Convert string to datetime
    new_date = date_obj + relativedelta(months=months)  # Add or subtract months
    return new_date.strftime("%Y-%m-%d")  # Convert back to string

def get_av_data(function, nm):
    fred = Fred(api_key="9ff6442c22eb1a25bb3ff361eceef633")
    data = fred.get_series(function, observation_start=add_months(STARTING_DATE, -1), observation_end=add_months(ENDING_DATE, 1))
    if len(data) > 0:   
        return pd.DataFrame(data.reset_index()).rename(columns={"index": "date", 0: nm}).fillna(method='ffill')
    else:
        return None

In [11]:
df = pd.read_csv("DDQN_Test_Cases.csv")
# predicted_labels = pd.read_csv("predicted_labels.csv")
# predicted_labels = predicted_labels[["date", "predicted_label"]]
# predicted_labels['date'] = pd.to_datetime(predicted_labels['date'], errors='coerce') 

for test_id in range(len(df)):
    for repeats in range(1):
        test_id = 0
        repeats = 1
        name = df["Name"][test_id]
        name = f"{name}_{repeats}"
        dataset_name = df["Dataset"][test_id]
        args = df["State"][test_id]
        sentype = df["Sentiment Type"][test_id]
        data = yf.download(dataset_name, start=STARTING_DATE, end=ENDING_DATE).reset_index().rename(columns={"Date": "date"})
        data.columns = data.columns.droplevel(1)
        data['LogReturn'] = np.log(data['Open'] / data['Open'].shift(1))
        data['Volatility'] = data['LogReturn'].ewm(span=45).std()        
        if len(data) <= 0:
            print("Error downloading financial data")
            sys.exit()
        sentiments = pd.read_csv(f"{sentype}.csv")[["date", "pos_avg", "neg_avg", "neu_avg"]]
        sentiments["compound_avg"] = sentiments["pos_avg"] - sentiments["neg_avg"]
        sentiments['date'] = pd.to_datetime(sentiments['date'])
        cpi = get_av_data("CPIAUCSL", 'cpi')
        if len(data) <= 0:
            print("Error downloading sentiment data")
            sys.exit()
        # datasets = [cpi, predicted_labels]
        datasets = [cpi, sentiments]
        merged_data = data.sort_values("date")
        for dataset in datasets:
            merged_data = pd.merge_asof(
                merged_data.sort_values("date"),
                dataset.sort_values("date"),
                on="date",
                direction="backward"
            )
        merged_data = merged_data.dropna()
        optimized = optimize_sharpe(merged_data, Fu_range, Fl_range, Vt_range)
        labels = assign_labels(merged_data, 1,1,1, optimized)
        merged_data["label"]  = merged_data.apply(lambda x: get_label_for_a_day(labels, x["date"]), axis=1)
        merged_data["prev_label"]  = merged_data.apply(lambda x: get_prev_label_for_a_day(labels, x["date"]), axis=1)
        merged_data = merged_data.dropna()
        # fig = go.Figure(data=[go.Candlestick(
        #     x=merged_data["date"],
        #     open=merged_data["Open"],
        #     high=merged_data["High"],
        #     low=merged_data["Low"],
        #     close=merged_data["Close"]
        # )])
        # draw_barriers(merged_data, fig, 1,1,1, optimized)
        # fig.show()

        tbl_dummies = pd.get_dummies(merged_data["label"], prefix="TBL")
        merged_data = pd.concat([merged_data, tbl_dummies], axis=1)

        prev_tbl_dummies = pd.get_dummies(merged_data["prev_label"], prefix="TBL_Prev")
        merged_data = pd.concat([merged_data, prev_tbl_dummies], axis=1)

        merged_data["SEN_Increasing"] = (merged_data["compound_avg"] > merged_data["compound_avg"].shift(1)).astype(int)
        merged_data["POS_Increasing"] = (merged_data["pos_avg"] > merged_data["pos_avg"].shift(1)).astype(int)
        merged_data["NEG_Increasing"] = (merged_data["neg_avg"] > merged_data["neg_avg"].shift(1)).astype(int)

        vol_threshold = merged_data['Volatility'].rolling(window=45).quantile(0.7)
        merged_data['Volatility_Low'] = (merged_data['Volatility'] < vol_threshold).astype(int)
        merged_data['Volatility_High'] = (merged_data['Volatility'] > vol_threshold).astype(int)

        merged_data["RSI"] = ta.rsi(merged_data["Close"], length=10)
        merged_data['RSI_Label'] = categorize_rsi(merged_data['RSI'])
        rsi_dummies = pd.get_dummies(merged_data['RSI_Label'], prefix='RSI')
        merged_data = pd.concat([merged_data, rsi_dummies], axis=1)

        merged_data["SEN_Above_Rolling"] = (merged_data["compound_avg"] > merged_data["compound_avg"].rolling(14).mean()).astype(int)
        merged_data["POS_Above_Rolling"] = (merged_data["pos_avg"] > merged_data["pos_avg"].rolling(14).mean()).astype(int)
        merged_data["NEG_Above_Rolling"] = (merged_data["neg_avg"] > merged_data["neg_avg"].rolling(14).mean()).astype(int)

        merged_data["ROC"] = merged_data["Close"].pct_change(periods=8) * 100
        roc_std = merged_data['ROC'].rolling(window=8).std()
        merged_data['ROC_Label'] = np.where(merged_data['ROC'] > roc_std, LABEL_BULLISH, np.where(merged_data['ROC'] < -roc_std, LABEL_BEARISH, LABEL_NEUTRAL))
        roc_dummies = pd.get_dummies(merged_data['ROC_Label'], prefix='ROC')
        merged_data = pd.concat([merged_data, roc_dummies], axis=1)

        merged_data["Close_Increasing"] = ((merged_data["Close"].shift(1) < merged_data["Close"]).astype(int)).shift(-1)        

#         merged_data['log_return'] = np.log(merged_data['Close'] / merged_data['Close'].shift(1))
#         merged_data['Close'] = merged_data['log_return'].ewm(span=252, adjust=False).std()
#         merged_data['Close'] = merged_data['LogReturn'] / merged_data['ewma_std']

#         merged_data["SMA_50"] = merged_data["Close"].rolling(window=50).mean()
#         merged_data["SMA_10"] = merged_data["Close"].rolling(window=10).mean()
#         merged_data["SMA_Above_50"] = (merged_data["Close"] > merged_data["SMA_50"]).astype(int)
#         merged_data["SMA_10_Increasing"] = (merged_data["SMA_10"].shift(1) < merged_data["SMA_10"]).astype(int)
#         merged_data["SMA_50_Increasing"] = (merged_data["SMA_50"].shift(1) < merged_data["SMA_50"]).astype(int)
#         merged_data["CPI_50"] = merged_data["cpi"].rolling(window=50).mean()
#         merged_data["CPI_Above_50"] = (merged_data["cpi"] > merged_data["CPI_50"]).astype(int)
#         merged_data["crossover"] = (merged_data["SMA_10"] > merged_data["SMA_50"]).astype(int)
#         merged_data["MACD"] = ta.macd(merged_data["Close"], length=50)['MACD_12_26_9']
#         merged_data["MACD_10"] = merged_data["MACD"].rolling(window=10).mean()
#         merged_data["MACD_Above_10"] = (merged_data["MACD"] > merged_data["MACD_10"]).astype(int)
#         merged_data["Signal_Line"] = merged_data["MACD"].ewm(span=9, adjust=False).mean()
#         merged_data["MACD_Bullish_Crossover"] = (merged_data["MACD"] > merged_data["Signal_Line"]).astype(int)
#         merged_data['pos_avg_norm'] = (merged_data['pos_avg'] - merged_data['pos_avg'].min()) / (merged_data['pos_avg'].max() - merged_data['pos_avg'].min())
#         merged_data['Medium_Positive'] = merged_data['pos_avg_norm'].apply(lambda x: x > (1/3) and x < (2/3)).astype(int)
#         merged_data['neg_avg_norm'] = (merged_data['neg_avg'] - merged_data['neg_avg'].min()) / (merged_data['neg_avg'].max() - merged_data['neg_avg'].min())
#         merged_data['Low_Negative'] = merged_data['neg_avg_norm'].apply(lambda x: x < (1/3)).astype(int)
#         merged_data["SEN_10"] = merged_data["compound_avg"].rolling(window=10).mean()
#         merged_data["SEN_Above_10"] = (merged_data["compound_avg"] > merged_data["SEN_10"]).astype(int)
#         merged_data["RSI"] = ta.rsi(merged_data["Close"], length=50)
#         merged_data["RSI_50"] = merged_data["RSI"].rolling(window=50).mean()
#         merged_data["RSI_10"] = merged_data["RSI"].rolling(window=10).mean()
#         merged_data["RSI_Above_10"] = (merged_data["RSI"] > merged_data["RSI_10"]).astype(int)
#         merged_data["RSI_Increasing"] = (merged_data["RSI"].shift(1) < merged_data["RSI"]).astype(int)
#         merged_data["ROC"] = ((merged_data["Close"] - merged_data["Close"].shift(5)) / merged_data["Close"].shift(5)) * 100
#         merged_data["ROC_50"] = merged_data["ROC"].rolling(window=50).mean()
#         merged_data["ROC_Above_50"] = (merged_data["ROC"] > merged_data["ROC_50"]).astype(int)
#         merged_data["ROC_Increasing"] = (merged_data["ROC"].shift(1) < merged_data["ROC"]).astype(int)
#         merged_data["Momentum"] = merged_data["Close"].diff(5)
#         merged_data["Momentum_50"] = merged_data["Momentum"].rolling(window=50).mean()
#         merged_data["Momentum_Above_50"] = (merged_data["Momentum"] > merged_data["Momentum_50"]).astype(int)
#         merged_data["Momentum_Increasing"] = (merged_data["Momentum"].shift(1) < merged_data["Momentum"]).astype(int)
#         merged_data["POS_Above_Mean"] = (merged_data["pos_avg"] > merged_data["pos_avg"].mean()).astype(int)
#         merged_data["SEN_Above_Mean"] = (merged_data["compound_avg"] > merged_data["compound_avg"].mean()).astype(int)
#         merged_data["RSI_Above_50"] = (merged_data["RSI"] > merged_data["RSI_50"]).astype(int)
#         merged_data["NEG_Below_Mean"] = (merged_data["neg_avg"] < merged_data["neg_avg"].mean()).astype(int)
#         merged_data['compound_avg_norm'] = (merged_data['compound_avg'] - merged_data['compound_avg'].min()) / (merged_data['compound_avg'].max() - merged_data['compound_avg'].min())
#         merged_data['High_Sentiment'] = merged_data['compound_avg_norm'].apply(lambda x: x > (2/3)).astype(int)

        merged_data = merged_data.dropna()
        merged_data = merged_data[("Close label "+args).split(" ")].values.tolist()

        split_index = int(len(merged_data) * 0.8)
        train_data =  merged_data[:split_index]
        test_data = merged_data[split_index:]
        scaler = MinMaxScaler()
        train_data_drop1st_column = [row[2:] for row in train_data]
        scaler.fit(train_data_drop1st_column)
        train_data_scaled = scaler.transform(train_data_drop1st_column).tolist()
        test_data_scaled = scaler.transform([row[2:] for row in test_data]).tolist()
        if len(train_data) > 0:
            dqn = DQNAlgorithm(data=train_data, data_scaled=train_data_scaled, episodes=EPISODES, gamma=GAMMA, epsilon=EPSILON, epsilon_min=EPSILON_MIN, epsilon_decay = EPSILON_DECAY)
            if TEST == True:
                dqn.test()
            else:
                training_start = time.time()
                best_model = dqn.run()
                training_end = time.time()
                training_time = training_start - training_end
                model_returns_train = evaluate_model(train_data, train_data_scaled, best_model)
                model_returns_test = evaluate_model(test_data, test_data_scaled, best_model)
                best_model.save(f"{name}_model.h5")

                columns = ["Test Name", "# Purchases", "# Sells", "Successes", "Fails", "Sortino Ratio", "Sharpe Ratio", "Maximum Drawdown", "Profit Factor", "Final Balance", "Profit Per Trade", "Win Rate", "Total Profits", "Cumulative Returns"]

                train_conc = [name] + list(model_returns_train.values())
                test_conc =  [name] + list(model_returns_test.values())

                pd.DataFrame([train_conc], columns=columns).to_csv(f"Train_Results.csv", mode="a", index=False, header=not os.path.exists("Train_Results.csv"))
                pd.DataFrame([test_conc], columns=columns).to_csv(f"Test_Results.csv", mode="a", index=False, header=not os.path.exists("Test_Results.csv"))

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Episode 1/2, Win Rate: 51.470588235294116, Max Drawdown: 23.373052367467622, Cum Returns: 80
Elapsed time: 36.069512605667114

Episode 2/2, Win Rate: 88.60294117647058, Max Drawdown: 8.13834202521886, Cum Returns: 2100
Elapsed time: 48.36659073829651

